# 🔁 Cycle Detection — Runnable Notebook

Companion to [`README.md`](README.md) and
[`11_cycle_detection_lesson.html`](11_cycle_detection_lesson.html).

The rule differs by graph type: **undirected** (visited non-parent) vs **directed** (edge to an on-stack vertex).

## 1. Undirected — DFS remembering your parent

In [ ]:
def has_cycle_undirected(n, adj):
    """adj[u] = neighbours. Visited neighbour that ISN'T your parent -> cycle."""
    seen = [False] * n
    def dfs(u, parent):
        seen[u] = True
        for v in adj[u]:
            if not seen[v]:
                if dfs(v, u):              # recurse, remembering we came from u
                    return True
            elif v != parent:              # visited AND not our parent -> cycle
                return True
        return False
    for s in range(n):                     # cover disconnected pieces
        if not seen[s] and dfs(s, -1):
            return True
    return False

triangle = {0: [1, 2], 1: [0, 2], 2: [0, 1]}      # 0-1-2-0
tree     = {0: [1], 1: [0, 2, 3], 2: [1], 3: [1]} # no cycle
print("triangle has cycle?", has_cycle_undirected(3, triangle))
print("tree has cycle?    ", has_cycle_undirected(4, tree))
assert has_cycle_undirected(3, triangle)
assert not has_cycle_undirected(4, tree)

## 2. Undirected — the Union-Find alternative
An edge whose endpoints already share a group closes a cycle.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                   # already together -> cycle
        self.parent[rb] = ra
        return True

def has_cycle_uf(n, edges):
    uf = UnionFind(n)
    return any(not uf.union(u, v) for u, v in edges)

print("triangle (UF):", has_cycle_uf(3, [(0, 1), (1, 2), (2, 0)]))
print("path (UF)    :", has_cycle_uf(3, [(0, 1), (1, 2)]))
assert has_cycle_uf(3, [(0, 1), (1, 2), (2, 0)])
assert not has_cycle_uf(3, [(0, 1), (1, 2)])

## 3. Directed — the 3-colour DFS (back edges)
White = unseen, Grey = on the recursion stack, Black = finished. Edge to a **grey** = cycle.

In [ ]:
def has_cycle_directed(n, adj):
    """adj[u] = vertices u points to. Edge to a GREY (on-stack) node -> cycle."""
    WHITE, GREY, BLACK = 0, 1, 2
    color = [WHITE] * n
    def dfs(u):
        color[u] = GREY                    # now on the recursion stack
        for v in adj[u]:
            if color[v] == GREY:           # back edge to an ancestor -> cycle
                return True
            if color[v] == WHITE and dfs(v):
                return True
        color[u] = BLACK                   # finished; leaves the stack
        return False
    return any(color[s] == WHITE and dfs(s) for s in range(n))

dir_cycle = {0: [1], 1: [2], 2: [0]}        # 0 -> 1 -> 2 -> 0
dag       = {0: [1, 2], 1: [2], 2: []}      # 0->1, 0->2, 1->2 (no cycle)
print("directed cycle?", has_cycle_directed(3, dir_cycle))
print("DAG has cycle? ", has_cycle_directed(3, dag))
assert has_cycle_directed(3, dir_cycle)
assert not has_cycle_directed(3, dag)

## 4. Why not just a 'visited' flag for directed graphs?
Reaching a **finished** node via another path is normal. Only an **on-stack** node means a loop.

In [ ]:
# In the DAG above, vertex 2 is reachable from BOTH 0 and 1.
# A naive "any revisit = cycle" check would falsely flag 0->2 after 1->2.
# The 3-colour method only flags edges to GREY (on-stack) nodes, so the DAG passes.
print("DAG correctly reports no cycle:", not has_cycle_directed(3, dag))
assert not has_cycle_directed(3, dag)

## ✅ Recap
| | Undirected | Directed |
|---|---|---|
| DFS rule | visited & **not parent** → cycle | edge to a **grey** (on-stack) node → cycle |
| No-DFS option | **Union-Find** on edges | **Kahn's** topo sort (output `< V`) |
| Cost | `O(V+E)` | `O(V+E)` |

- Undirected: don't flag the edge back to your **parent**.
- Directed: don't flag edges to **finished (black)** nodes — only **grey** means a loop.

That closes the advanced graph track. See [`README`](../README.md) for the whole map.